# 01 - Data cleaning (Deliverable 1, part 1)
Runs `src/cleaning.py`: strips quantities/units/prep words/brands, lowercases, lemmatizes, merges near-duplicates (rapidfuzz), maps ingredients to flavor compounds. Outputs `data/processed/recipes.csv` and `ingredient_compounds.csv`.

In [1]:
import sys; sys.path.insert(0, '../src')
import cleaning, pandas as pd
cleaning.main()

39774 recipes, 6714 raw ingredient strings


recipes kept: 39752 / 39774
raw ingredient strings: 6714 -> cleaned unique: 5565
match rate (unique ingredients): 2927/5565 = 52.6%
match rate (by occurrence):      327314/426305 = 76.8%
match types: {'suffix': 1354, 'contains': 1184, 'exact': 365, 'alias': 13, 'fuzzy': 11}
top 40 unmatched (name, count): [('salt', 18050), ('water', 8679), ('kosher salt', 3113), ('oil', 1970), ('baking powder', 1738), ('jalapeno chilies', 1730), ('cooking spray', 1490), ('canola oil', 1223), ('sauce', 1126), ('salsa', 973), ('baking soda', 942), ('sea salt', 940), ('green chilies', 932), ('garam masala', 926), ('mayonaise', 852), ('curry powder', 709), ('worcestershire sauce', 702), ('coarse salt', 578), ('tofu', 547), ('caper', 541), ('allspice', 519), ('mirin', 503), ('eggplant', 501), ('cooking oil', 483), ('beansprouts', 454), ('black peppercorn', 453), ('hoisin sauce', 429), ('confectioner sugar', 395), ('italian seasoning', 395), ('cajun seasoning', 377), ('chile', 376), ('cornmeal', 368), ('half

## Cleaning examples (raw -> clean)

In [2]:
m = pd.read_csv('../data/processed/ingredient_map.csv')
m[m.raw != m.clean].sample(15, random_state=1)[['raw','clean','count']]

,raw,clean,count
3732,Domino Light Brown Sugar,domino light brown sugar,3
6041,fresh white truffles,white truffle,1
1987,whole wheat pita,wheat pita,14
4157,whole wheat angel hair pasta,wheat angel hair pasta,3
2802,reduced-sodium tamari sauce,tamari sauce,7
1234,nonfat ricotta cheese,ricotta cheese,35
401,unsweetened coconut milk,coconut milk,191
6013,licor 43,licor,1
4244,breast of lamb,breast lamb,2
370,thyme sprigs,thyme sprig,209


## Compound match quality by method

In [3]:
ic = pd.read_csv('../data/processed/ingredient_compounds.csv')
print(ic.groupby('match_type').agg(unique=('ingredient','count'), recipe_occurrences=('recipe_count','sum')))
ic[ic.match_type.isin(['suffix','contains'])].sample(12, random_state=2)[['ingredient','flavor_name','match_type']]

            unique  recipe_occurrences
match_type                            
alias           13               22503
contains      1184               40682
exact          365              181319
fuzzy           11                1184
suffix        1354               81626


,ingredient,flavor_name,match_type
828,medjool date,date,suffix
661,lemon pepper,pepper,suffix
1612,center cut pork loin chop,pork,contains
2861,lipton pure leaf unsweeten iced tea,tea,suffix
1236,chicken breast fillet,chicken,contains
1095,no salt added chicken broth,chicken broth,suffix
585,roast red pepper drain,pepper,contains
375,kernel corn drain,corn,contains
1689,beef round,beef,contains
2434,tomato jam,tomato,contains


**Reading the numbers:** 76.8% of ingredient occurrences (52.6% of unique names) are matched. `suffix`/`contains` matches fall back to the head noun (e.g. *chicken breast* -> *chicken*) - a deliberate approximation. Unmatched items are rare long-tail names plus staples absent from the Ahn et al. data (salt, water, oil, baking powder/soda). Farnesol is excluded (dataset README: it is both an ingredient and a compound).